# Notebook 13 — F3 (forecasting) NARRATIVE: the integrated null

**V1-S15 integration. No new compute, no model, read-only.** This notebook does **not** retrain,
rescore, or recompute the Gate G4 verdict (frozen in `data/v1/forecasting_test_wilcoxon.json`).
It **reads** the already-committed V1-S14 sealed-test artifacts and **re-frames** the signed
NULL (Gate G4) inside the integrated SciField story. Every number it cites is read from the
parquet/JSON artifacts, not hand-typed.

## The one-line claim

**Temporal/feature dynamics alone carry the emergence signal; the citation-graph topology adds
nothing.** On the sealed test set (forecast-ORIGIN years 2021–2022, n=138, 13 positives ≈ 9.4%),
the **graph-free `no_graph`** model — which consumes the node features *without* graph
message-passing — is the **best** test-set model (emergence AUC **0.804**). Adding the
heterogeneous citation graph (**HGT**, AUC **0.781**) does **not** help (H1 −**2.34pp**, bar was
+5pp) and is **significantly worse-calibrated** (it over-predicts; paired Brier-loss Wilcoxon
p≈2.63e-11, direction = favors_baseline). `overall_pass = False`. F3 is a **characterized null**.

## How F3 threads to F1 (two honest nulls about temporal structure)

Both F1 and F3 are statements about a topic's **temporal evolution**:

- **F1 (evidence-quality ↔ volume cascade) = NULL.** Across the corpus there is **no**
  population-level lead/lag cascade between a topic's evidence-quality trajectory and its
  publication-volume trajectory: **0 / 138** qualifying topics are directional, and the panel
  tests are non-significant (quality-leads p≈0.28, quality-lags p≈0.74).
- **F3 (graph-vs-graph-free emergence forecasting) = NULL.** The citation-graph *structure*
  adds nothing beyond temporal/feature dynamics for 3-year emergence.

Stated plainly: **two honest nulls about temporal structure** — F1 finds no temporal
lead/lag coupling, F3 finds graph topology adds nothing to temporal forecasting — while **F2
(dual-novelty) is the finding that holds**. Reporting both nulls cleanly, with characterized
mechanisms, is the honest integrated result.

> Provenance: Gate G4 signed NULL (`docs/gates/G4_forecasting.md`, Samer, 2026-06-05);
> OSF pre-registration PR2 DOI `10.17605/OSF.IO/XP94F`. The V1-S14 evaluation notebook is
> `notebooks/10_forecasting_eval.ipynb`; this notebook re-uses its conventions and references its
> figure `docs/figures/F3_forecasting.png` (which it does **not** overwrite).

## 1. Setup (read-only)

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

from scifield.repro import record_run  # noqa: E402


def _load_parquet(path: Path) -> pd.DataFrame | None:
    """Defensive parquet read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING artifact: {path} — cannot render this section.")
        return None
    return pd.read_parquet(path)


def _load_json(path: Path) -> dict | None:
    """Defensive JSON read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING json: {path} — fields unavailable.")
        return None
    return json.loads(path.read_text())


# F3 sealed-test artifacts (read-only; produced once by V1-S14 gnn-eval).
metrics = _load_parquet(DATA / "forecasting_test_metrics.parquet")
sens = _load_parquet(DATA / "forecasting_test_sensitivity.parquet")
wilcoxon = _load_json(DATA / "forecasting_test_wilcoxon.json")
metrics_sidecar = _load_json(DATA / "forecasting_test_metrics.parquet.run.json")

# F1 cascade verdict (read-only) — for the F1↔F3 temporal-evolution thread.
f1 = _load_json(DATA / "f1_cascade_verdict.json")

print(
    "loaded:",
    {
        k: ("ok" if v is not None else None)
        for k, v in {
            "metrics": metrics,
            "sens": sens,
            "wilcoxon": wilcoxon,
            "metrics_sidecar": metrics_sidecar,
            "f1": f1,
        }.items()
    },
)

loaded: {'metrics': 'ok', 'sens': 'ok', 'wilcoxon': 'ok', 'metrics_sidecar': 'ok', 'f1': 'ok'}


## 2. Confirmatory re-read — the 5-model sealed-test table

The graph-free `no_graph` model wins on emergence AUC; HGT trails it. The numbers below are read
straight from `forecasting_test_metrics.parquet` so this notebook reproduces what it cites.

In [2]:
N_TEST = int(metrics["n_test"].iloc[0])
N_TEST_POS = int(metrics["n_test_pos"].iloc[0])
N_TRAIN = int(metrics["n_train"].iloc[0])
POS_RATE = N_TEST_POS / N_TEST
print(
    f"Sealed test set: forecast-ORIGIN years 2021-2022 | n_train={N_TRAIN} "
    f"n_test={N_TEST} n_test_pos={N_TEST_POS} (pos rate {POS_RATE:.1%})"
)
print(
    f"Small-sample caveat: only {N_TEST_POS} positives — per-model AUCs are high-variance "
    "(this does NOT rescue F3; see G4 §Recommendation)."
)

order = ["naive", "arima", "mlp", "no_graph", "hgt"]
tbl = metrics.set_index("model").reindex(order)[["emergence_auc", "share_mape"]].round(4)
print()
print("5-model sealed-test table (emergence AUC; share MAPE):")
print(tbl.to_string())

auc = metrics.set_index("model")["emergence_auc"]
auc_hgt = float(auc["hgt"])
auc_ng = float(auc["no_graph"])
best_model = auc.idxmax()
print()
print(f"BEST test-set model by emergence AUC: '{best_model}' ({float(auc[best_model]):.4f}).")
print(
    f"-> the graph-FREE model leads; HGT ({auc_hgt:.4f}) trails it. "
    "Temporal/feature dynamics carry the signal."
)
tbl

Sealed test set: forecast-ORIGIN years 2021-2022 | n_train=1298 n_test=138 n_test_pos=13 (pos rate 9.4%)
Small-sample caveat: only 13 positives — per-model AUCs are high-variance (this does NOT rescue F3; see G4 §Recommendation).

5-model sealed-test table (emergence AUC; share MAPE):
          emergence_auc  share_mape
model                              
naive            0.4074      0.2605
arima            0.6529      0.2744
mlp              0.7009      0.3954
no_graph         0.8043      0.4610
hgt              0.7809      0.5729

BEST test-set model by emergence AUC: 'no_graph' (0.8043).
-> the graph-FREE model leads; HGT (0.7809) trails it. Temporal/feature dynamics carry the signal.


,emergence_auc,share_mape
model,,
naive,0.4074,0.2605
arima,0.6529,0.2744
mlp,0.7009,0.3954
no_graph,0.8043,0.4610
hgt,0.7809,0.5729


## 3. The graph contribution is on the wrong side of zero (PR2 §6 ablation + H1/H2)

The apples-to-apples "what does adding graph structure buy you" contrast is HGT vs. `no_graph`
(same target, same horizon, with vs. without graph message-passing). It is **negative**, and the
paired Brier-loss Wilcoxon (H2) is significant **against** HGT. Both read from the frozen
`forecasting_test_wilcoxon.json` verdict block.

In [3]:
pb = wilcoxon["primary_brier"]
rs = wilcoxon["secondary_raw_score"]
vd = wilcoxon["verdict"]

delta_pp = float(vd["delta_pp"])  # frozen in the verdict block
margin_pp = float(vd["margin_pp"])
ref_bar = auc_ng + margin_pp / 100.0  # the +5pp-over-best-baseline bar HGT had to clear

print("ABLATION (PR2 §6, HGT minus graph message-passing):")
print(f"  HGT emergence AUC      = {auc_hgt:.4f}")
print(f"  no_graph emergence AUC = {auc_ng:.4f}  (val-selected best baseline)")
print(f"  delta (HGT - no_graph) = {delta_pp:+.2f} pp   <- on the WRONG SIDE OF ZERO")
print(f"  +{margin_pp:.0f}pp pre-registered bar = {ref_bar:.4f}; HGT would have needed >= that.")
print()
print("H1  (delta > +5pp AUC):  ", vd["h1_pass"], f"  (delta = {delta_pp:+.2f} pp)")
print(
    "H2  (paired Brier p<0.05):",
    vd["h2_pass"],
    f"  p={float(pb['pvalue']):.3e}  direction={pb['direction'].upper()}",
)
print(
    f"     primary Brier: statistic={pb['statistic']:.1f}  n_pairs={pb['n_pairs']}  "
    f"median(brier_hgt - brier_ng)={float(pb['median_diff']):+.4f}"
)
print(
    f"     secondary raw-score: p={float(rs['pvalue']):.3e}  direction={rs['direction']} "
    "(HGT emits systematically HIGHER scores -> over-prediction)"
)
print()
print(f"  overall_pass = {vd['overall_pass']}  ->  {vd['mechanical_recommendation']}")
print("  Reading: H2 is mechanically p<0.05 but FAVORS THE BASELINE (HGT worse-calibrated),")
print("  so its significance REINFORCES the null rather than rescuing F3. Clean FAIL/NULL, not a")
print("  near-miss — HGT sits on the wrong side of the bar.")

ABLATION (PR2 §6, HGT minus graph message-passing):
  HGT emergence AUC      = 0.7809
  no_graph emergence AUC = 0.8043  (val-selected best baseline)
  delta (HGT - no_graph) = -2.34 pp   <- on the WRONG SIDE OF ZERO
  +5pp pre-registered bar = 0.8543; HGT would have needed >= that.

H1  (delta > +5pp AUC):   False   (delta = -2.34 pp)
H2  (paired Brier p<0.05): True   p=2.629e-11  direction=FAVORS_BASELINE
     primary Brier: statistic=1659.0  n_pairs=138  median(brier_hgt - brier_ng)=+0.2330
     secondary raw-score: p=2.156e-24  direction=model_higher (HGT emits systematically HIGHER scores -> over-prediction)

  overall_pass = False  ->  NULL FINDING (F3 reported as null)
  Reading: H2 is mechanically p<0.05 but FAVORS THE BASELINE (HGT worse-calibrated),
  so its significance REINFORCES the null rather than rescuing F3. Clean FAIL/NULL, not a
  near-miss — HGT sits on the wrong side of the bar.


## 4. Mechanism: HGT over-predicts (worst-in-class share-MAPE)

One mechanism drives both adverse outcomes. HGT assigns 0.4–0.8 emergence probability to many
units whose realized emergence is ~0, while the base rate is ≈9.4%. `no_graph` keeps its mass near
the base rate and is roughly calibrated. The over-prediction shows up directly as HGT's
**worst-in-class** share-forecast error (`share_mape`), which we read from the metrics table.

In [4]:
smape = metrics.set_index("model")["share_mape"].reindex(order)
worst_smape_model = smape.idxmax()
print("share_mape (lower = better) by model:")
for m in order:
    flag = "  <- WORST (over-prediction)" if m == worst_smape_model else ""
    print(f"  {m:>9}: {float(smape[m]):.4f}{flag}")
print()
print(
    f"HGT has the worst share-MAPE ({float(smape['hgt']):.4f}); no_graph mass stays near the "
    f"{POS_RATE:.1%} base rate."
)
print("The same over-prediction explains the losing paired Brier comparison in §3 — see the")
print("reliability panel in docs/figures/F3_forecasting.png (V1-S14; not overwritten here).")

share_mape (lower = better) by model:
      naive: 0.2605
      arima: 0.2744
        mlp: 0.3954
   no_graph: 0.4610
        hgt: 0.5729  <- WORST (over-prediction)

HGT has the worst share-MAPE (0.5729); no_graph mass stays near the 9.4% base rate.
The same over-prediction explains the losing paired Brier comparison in §3 — see the
reliability panel in docs/figures/F3_forecasting.png (V1-S14; not overwritten here).


## 5. The null is robust to the label definition (descriptive)

Sensitivity is descriptive only — it cannot convert a fail into a pass (PR2 §9 / Gate G4) — but it
shows the null is not a labeling artifact: HGT AUC is no higher (indeed lower) under every
alternate pre-registered label, never approaching its own +5pp bar.

In [5]:
print("HGT emergence AUC under alternate pre-registered labels (descriptive robustness):")
print(sens.round(4).to_string(index=False))
print()
print("Every variant is <= primary; the null is robust to the label definition.")
sens.round(4)

HGT emergence AUC under alternate pre-registered labels (descriptive robustness):
      variant         label_column  emergence_auc  n_test  n_test_pos  delta_vs_primary_pp
      primary             emergent         0.7809     138          13               0.0000
additive_jump    emergent_additive         0.6315     138          30             -14.9442
  count_surge emergent_count_surge         0.7068     138           5              -7.4156

Every variant is <= primary; the null is robust to the label definition.


,variant,label_column,emergence_auc,n_test,n_test_pos,delta_vs_primary_pp
0,primary,emergent,0.7809,138,13,0.0000
1,additive_jump,emergent_additive,0.6315,138,30,-14.9442
2,count_surge,emergent_count_surge,0.7068,138,5,-7.4156


## 6. The F1↔F3 thread: two honest nulls about temporal structure

F3 is one of **two** temporal-evolution nulls in the integrated story. F1 asked whether a topic's
**evidence-quality** trajectory leads/lags its **publication-volume** trajectory (a temporal
cascade); the answer across the corpus is **no**. Below we read the F1 verdict so the connection is
stated from the artifact, not from memory.

In [6]:
f1p = f1["primary"]
print("F1 — evidence-quality <-> volume cascade (primary, leaf topics):")
print(f"  f1_verdict           = {f1['f1_verdict']}")
print(
    f"  directional topics   = {f1p['n_directional']} / {f1['n_qualifying']} "
    f"(frac {f1p['frac_directional']:.3f}; threshold {f1p['frac_threshold']:.2f}); "
    f"holds={f1p['holds']}"
)
print(
    f"  panel quality-leads p = {f1p['panel_p_quality_leads']:.3f}  |  "
    f"quality-lags p = {f1p['panel_p_quality_lags']:.3f}  (both non-sig)"
)
print()
print("F3 — graph-vs-graph-free 3-yr emergence forecasting (sealed test):")
print(
    f"  verdict              = {vd['mechanical_recommendation']}  "
    f"(overall_pass={vd['overall_pass']})"
)
print(
    f"  graph contribution   = {delta_pp:+.2f} pp AUC (HGT - no_graph); "
    f"H1 bar was +{margin_pp:.0f}pp"
)
print()
print("THREAD (state this plainly in the results draft):")
print("  Both F1 and F3 concern a topic's TEMPORAL EVOLUTION, and both are NULL:")
print("   - F1: no population-level evidence-quality<->volume lead/lag cascade.")
print("   - F3: citation-graph structure adds nothing beyond temporal/feature dynamics.")
print(
    "  => TWO honest nulls about temporal structure; "
    "F2 (dual-novelty) is the finding that holds."
)

F1 — evidence-quality <-> volume cascade (primary, leaf topics):
  f1_verdict           = NULL
  directional topics   = 0 / 138 (frac 0.000; threshold 0.20); holds=False
  panel quality-leads p = 0.277  |  quality-lags p = 0.739  (both non-sig)

F3 — graph-vs-graph-free 3-yr emergence forecasting (sealed test):
  verdict              = NULL FINDING (F3 reported as null)  (overall_pass=False)
  graph contribution   = -2.34 pp AUC (HGT - no_graph); H1 bar was +5pp

THREAD (state this plainly in the results draft):
  Both F1 and F3 concern a topic's TEMPORAL EVOLUTION, and both are NULL:
   - F1: no population-level evidence-quality<->volume lead/lag cascade.
   - F3: citation-graph structure adds nothing beyond temporal/feature dynamics.
  => TWO honest nulls about temporal structure; F2 (dual-novelty) is the finding that holds.


## 7. Integrated summary panel — `docs/figures/F3_forecasting_narrative.png`

ONE panel for the integrated figure set: the 5-model test-AUC bars with the +5pp-over-`no_graph`
bar drawn and HGT shown failing it. The caption carries the integrated message — *graph adds
nothing; temporal dynamics suffice* — and notes **F1 + F3 as the two temporal-evolution nulls**.
This is a NEW narrative figure; it does **not** overwrite the V1-S14 diagnostic figure
`docs/figures/F3_forecasting.png`.

In [7]:
auc_by = metrics.set_index("model")["emergence_auc"].reindex(order)
# Grey baselines/MLP; blue = graph-free winner; red = HGT (the graph model that fails).
colors = ["#9aa0a6", "#9aa0a6", "#9aa0a6", "#4285f4", "#ea4335"]

fig, ax = plt.subplots(figsize=(8.2, 5.0))
bars = ax.bar(order, auc_by.values, color=colors, edgecolor="white", zorder=3)

# +5pp-over-no_graph pre-registered bar that HGT had to clear.
ax.axhline(ref_bar, ls="--", c="k", lw=1.4, zorder=4)
ax.text(
    -0.42,
    ref_bar + 0.006,
    f"+{margin_pp:.0f}pp-over-no_graph bar = {ref_bar:.3f}  (H1 threshold)",
    fontsize=8.5,
    fontweight="bold",
)
ax.axhline(0.5, ls=":", c="#bbbbbb", lw=1, zorder=1)
ax.text(-0.42, 0.5 + 0.004, "chance = 0.50", fontsize=8, color="#888888")

# Value labels on each bar.
for b, v in zip(bars, auc_by.values, strict=True):
    ax.text(
        b.get_x() + b.get_width() / 2,
        v + 0.008,
        f"{v:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Annotate the HGT shortfall vs. the winner.
ax.annotate(
    f"HGT {delta_pp:+.2f}pp vs no_graph\n(graph adds nothing)",
    xy=(4, auc_hgt),
    xytext=(3.05, auc_hgt - 0.135),
    fontsize=8.5,
    ha="center",
    color="#ea4335",
    arrowprops=dict(arrowstyle="->", color="#ea4335", lw=1.2),
)

ax.set_ylim(0.35, 0.92)
ax.set_ylabel("emergence AUC (sealed test)")
ax.set_xlabel("forecasting model  (grey = baselines/MLP · blue = graph-FREE · red = HGT graph)")
ax.set_title(
    "F3 — graph topology adds nothing to 3-yr emergence forecasting (Gate G4: NULL)\n"
    f"graph-free no_graph wins ({auc_ng:.3f}); HGT {auc_hgt:.3f} fails the +{margin_pp:.0f}pp bar; "
    f"Brier favors baseline (p={float(pb['pvalue']):.1e})",
    fontsize=10,
)
ax.tick_params(axis="x", rotation=12)
ax.grid(axis="y", ls=":", c="#e6e6e6", zorder=0)

# Integrated-story footnote: F1 + F3 are the two temporal-evolution nulls; F2 holds.
fig.text(
    0.5,
    -0.02,
    "Integrated story: F1 (no quality↔volume temporal cascade) + F3 (graph adds nothing to "
    "temporal forecasting) = two honest nulls about temporal structure; F2 (dual-novelty) holds.",
    ha="center",
    fontsize=8,
    style="italic",
    color="#444444",
)

fig.tight_layout()
F3N_FIG = FIGURES_DIR / "F3_forecasting_narrative.png"
fig.savefig(F3N_FIG, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print("wrote", F3N_FIG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/F3_forecasting_narrative.png


In [8]:
# Enforce the < 1 MB figure budget.
sz = F3N_FIG.stat().st_size
print(f"F3 narrative size = {sz / 1024:.1f} KB  (dpi={DPI})")
assert sz < 1_000_000, f"figure too large: {sz} bytes"
print("size assertion PASS — under 1 MB")

F3 narrative size = 76.3 KB  (dpi=120)
size assertion PASS — under 1 MB


In [9]:
# Provenance sidecar — fold in the source metrics artifact's config_hash + git_sha so the
# narrative figure is traceable back to the V1-S14 gnn-eval run.
src_hash = metrics_sidecar["config_hash"] if metrics_sidecar else None
src_sha = metrics_sidecar["git_sha"] if metrics_sidecar else None
sidecar_out = record_run(
    artifact_path=F3N_FIG,
    inputs={
        "metrics": DATA / "forecasting_test_metrics.parquet",
        "wilcoxon": DATA / "forecasting_test_wilcoxon.json",
    },
    config={
        "figure": "F3_forecasting_narrative",
        "session": "V1-S15",
        "role": "integration_narrative",
        "dpi": DPI,
        "n_test": N_TEST,
        "n_test_pos": N_TEST_POS,
        "auc_hgt": auc_hgt,
        "auc_no_graph": auc_ng,
        "best_model": str(best_model),
        "ablation_delta_pp": delta_pp,
        "h1_bar_pp": margin_pp,
        "brier_pvalue": float(pb["pvalue"]),
        "brier_direction": pb["direction"],
        "overall_pass": bool(vd["overall_pass"]),
        "f1_verdict": str(f1["f1_verdict"]),
        "f1_n_directional": int(f1p["n_directional"]),
        "f1_n_qualifying": int(f1["n_qualifying"]),
        "narrative": "two_temporal_evolution_nulls_F1_F3_F2_holds",
        "source_metrics_config_hash": src_hash,
        "source_metrics_git_sha": src_sha,
    },
)
print("recorded run sidecar:", sidecar_out)
print("figure exists:", F3N_FIG.exists(), "| KB:", f"{F3N_FIG.stat().st_size / 1024:.1f}")

recorded run sidecar: /Users/samersalman/Desktop/SciField/docs/figures/F3_forecasting_narrative.png.run.json
figure exists: True | KB: 76.3


## 8. Summary for the integrated narrative

**F3 framing (committed):** On the sealed test set, **temporal/feature dynamics alone carry the
3-year emergence signal** — the graph-free `no_graph` model is the best test-set model
(AUC **0.804**), while adding the citation-graph topology (**HGT**, AUC **0.781**) does **not**
help (**−2.34pp**, H1 FAIL vs. a +5pp bar) and is **significantly worse-calibrated**
(over-predicts; paired Brier-loss Wilcoxon **p≈2.63e-11**, direction = **favors_baseline**).
F3 is a **characterized null**: the citation graph adds no test-set forecasting value beyond the
temporal/structural node features.

**F1↔F3 thread (for downstream consistency):** F1 and F3 are **two honest nulls about a topic's
temporal evolution** — F1 finds no population-level evidence-quality↔volume lead/lag cascade
(0/138 directional; panel non-sig), and F3 finds the citation-graph structure adds nothing beyond
temporal dynamics for emergence. **F2 (dual-novelty) is the finding that holds.** Report both
nulls cleanly, with their characterized mechanisms (F1: no coupling; F3: HGT over-prediction).

Figures: integrated narrative panel `docs/figures/F3_forecasting_narrative.png` (this notebook);
the V1-S14 3-panel diagnostic `docs/figures/F3_forecasting.png` (unchanged) remains the detailed
AUC / reliability / paired-Brier view. Gate of record: `docs/gates/G4_forecasting.md`
(signed NULL, OSF DOI XP94F).